# Double JPEG Compression Detection  # DIP project 
# -GROUP 3
Laxman Patel 12341320,
Purva JIvani 12341010,
Harshwardhan Kamble 12340930,
Tanishq Gupta 12342220,


In [ ]:
import subprocess, sys
for pkg in ["torch", "torchvision", "scipy", "scikit-learn", "Pillow", "matplotlib", "numpy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)
print("Dependencies ready.")


Dependencies ready.


In [ ]:
import os

DATASET_PATH           = r"\data"   ##datsetpath  for tif images 
QUALITY_FACTORS_MAIN   = [20, 40, 60, 70, 75, 80, 90]
QUALITY_FACTORS_DETAIL = [60, 70, 75, 80, 85, 90]
RANDOM_SEED            = 42
NUM_ITERATIONS         = 10
BATCH_SIZE             = 64       
LEARNING_RATE          = 1e-3
EPOCHS                 = 60
CHECKPOINT_DIR         = r"\checkpoints" ##checkpoint for model
PLOTS_DIR              = r"\plots"   ##outputs
BEST_MODELS_DIR        = r"\best_models" ##best models so far 

for d in (CHECKPOINT_DIR, PLOTS_DIR, BEST_MODELS_DIR):
    os.makedirs(d, exist_ok=True)

print("Config OK. Dirs created.")


Config OK. Dirs created.


In [ ]:
# IMPORTS
import json, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from scipy.fftpack import dct as sp_dct, idct as sp_idct
from sklearn.metrics import confusion_matrix
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")


PyTorch device: cpu


In [ ]:
def _Q(q: int) -> np.ndarray:
    base = np.array([16,11,10,16,24,40,51,61,
                     12,12,14,19,26,58,60,55,
                     14,13,16,24,40,57,69,56,
                     14,17,22,29,51,87,80,62,
                     18,22,37,56,68,109,103,77,
                     24,35,55,64,81,104,113,92,
                     49,64,78,87,103,121,120,101,
                     72,92,95,98,112,100,103,99], np.float64).reshape(8, 8)
    sc = 5000 / q if q < 50 else 200 - 2 * q
    return np.clip(np.floor((base * sc + 50) / 100), 1, 255)

_Q_CACHE = {q: _Q(q) for q in set(QUALITY_FACTORS_MAIN + QUALITY_FACTORS_DETAIL)}


def _dct2_batch(blocks: np.ndarray) -> np.ndarray:
        return sp_dct(sp_dct(blocks, axis=2, norm='ortho'), axis=1, norm='ortho')


def _idct2_batch(blocks: np.ndarray) -> np.ndarray:
    return sp_idct(sp_idct(blocks, axis=2, norm='ortho'), axis=1, norm='ortho')


def extract_image_blocks_vectorised(gray: np.ndarray, q: int):
    Q = _Q_CACHE[q]
    H, W = gray.shape
    H8, W8 = (H // 8) * 8, (W // 8) * 8
    g = gray[:H8, :W8].astype(np.float64)
    N_r, N_c = H8 // 8, W8 // 8
    blocks = g.reshape(N_r, 8, N_c, 8).transpose(0, 2, 1, 3).reshape(-1, 8, 8)

    # Single compression
    dct1  = _dct2_batch(blocks)
    qdct1 = np.round(dct1 / Q)
    sp1   = np.clip(np.round(_idct2_batch(qdct1 * Q)), 0, 255)

    # Double compression
    qdct2 = np.round(_dct2_batch(sp1) / Q)
    sp2   = np.clip(np.round(_idct2_batch(qdct2 * Q)), 0, 255)

    return qdct1, qdct2, sp1, sp2, Q, len(blocks)


def compute_stability_batch(qdct_n: np.ndarray, Q: np.ndarray,
                             max_s: int = 5) -> np.ndarray:
 
    stability = np.full(len(qdct_n), max_s, dtype=np.int32)
    settled   = np.zeros(len(qdct_n), dtype=bool)

    sp = np.clip(np.round(_idct2_batch(qdct_n * Q)), 0, 255)
    q1 = np.round(_dct2_batch(sp) / Q)

    mask0 = np.all(qdct_n == q1, axis=(1, 2))
    stability[mask0] = 0
    settled[mask0]   = True

    prev    = q1.copy()
    sp_prev = np.clip(np.round(_idct2_batch(prev * Q)), 0, 255)

    for s in range(1, max_s + 1):
        active = ~settled
        if not active.any():
            break  

        nxt   = np.round(_dct2_batch(sp_prev[active]) / Q)
        same  = np.all(prev[active] == nxt, axis=(1, 2))
        newly = np.where(active)[0][same]
        stability[newly] = s
        settled[newly]   = True

        still_active = active.copy()
        still_active[newly] = False
        if not still_active.any():
            break  

        prev[still_active]    = np.round(_dct2_batch(sp_prev[still_active]) / Q)
        sp_prev[still_active] = np.clip(
            np.round(_idct2_batch(prev[still_active] * Q)), 0, 255)

    return stability


def compute_error_blocks_batch(qdct_n: np.ndarray, Q: np.ndarray,
                                n_stages: int = 3):
   
    N = len(qdct_n)
    if N == 0:
        empty = np.zeros((0, 8, 8), dtype=np.float32)
        return [empty] * n_stages, [empty] * n_stages

    error_list, dct_error_list = [], []
    qc = qdct_n.copy()
    sp = np.clip(np.round(_idct2_batch(qc * Q)), 0, 255)

    for _ in range(n_stages):
        idct_v = _idct2_batch(qc * Q)
        err    = np.clip(np.round(idct_v), 0, 255) - idct_v
        error_list.append(err)
        dct_error_list.append(_dct2_batch(err))
        qc = np.round(_dct2_batch(sp) / Q)
        sp = np.clip(np.round(_idct2_batch(qc * Q)), 0, 255)

    return error_list, dct_error_list


print("JPEG utilities defined.")


JPEG utilities defined.


In [ ]:
#  DATASET BUILDING


def _process_one_image(path, q):
    
    try:
        gray = np.array(Image.open(path).convert('L'), dtype=np.float64)
    except Exception:
        return None

    qdct1, qdct2, _, _, Q, _ = extract_image_blocks_vectorised(gray, q)
    s_single = compute_stability_batch(qdct1, Q)
    s_double = compute_stability_batch(qdct2, Q)
    mask_s   = s_single >= 1
    mask_d   = s_double >= 1

    if not (mask_s.any() or mask_d.any()):
        return None
    
    if mask_s.any():
        errs_s, derrs_s = compute_error_blocks_batch(qdct1[mask_s], Q)
    else:
        errs_s = derrs_s = [np.zeros((0, 8, 8), np.float32)] * 3

    if mask_d.any():
        errs_d, derrs_d = compute_error_blocks_batch(qdct2[mask_d], Q)
    else:
        errs_d = derrs_d = [np.zeros((0, 8, 8), np.float32)] * 3

    Es   = np.stack(errs_s,  axis=-1).astype(np.float32)
    Eds  = np.stack(derrs_s, axis=-1).astype(np.float32)
    Ed2  = np.stack(errs_d,  axis=-1).astype(np.float32)
    Edd2 = np.stack(derrs_d, axis=-1).astype(np.float32)

    stab_s = s_single[mask_s]
    stab_d = s_double[mask_d]
    rnd_s  = (np.all(np.abs(errs_s[0]) <= 0.5, axis=(1, 2))
               if mask_s.any() else np.array([], dtype=bool))
    rnd_d  = (np.all(np.abs(errs_d[0]) <= 0.5, axis=(1, 2))
               if mask_d.any() else np.array([], dtype=bool))

    Ein   = np.concatenate([Es,   Ed2],   axis=0)
    Edin  = np.concatenate([Eds,  Edd2],  axis=0)
    lbls  = np.concatenate([np.zeros(len(Es),  np.int64),
                             np.ones(len(Ed2), np.int64)])
    stabs = np.concatenate([stab_s, stab_d])
    rnds  = np.concatenate([rnd_s,  rnd_d])
    return Ein, Edin, lbls, stabs, rnds


def build_dataset_cached(image_paths, q, tag="train"):
    
    cache_path = os.path.join(CHECKPOINT_DIR, f"cache_{tag}_q{q}.npz")
    if os.path.exists(cache_path):
        print(f"  [cache] {cache_path}")
        d = np.load(cache_path)
        return d['Ein'], d['Edin'], d['labels'], d['stabs'], d['rnds']

    print(f"  Extracting {tag} blocks q={q}  ({len(image_paths)} images) ...", end="", flush=True)
    t0 = time.time()
    
    parts = [_process_one_image(p, q) for p in image_paths]
    parts = [r for r in parts if r is not None]
    if not parts:
        raise RuntimeError(f"No valid blocks for q={q}, tag={tag}")

    Ein   = np.concatenate([r[0] for r in parts], axis=0)
    Edin  = np.concatenate([r[1] for r in parts], axis=0)
    lbls  = np.concatenate([r[2] for r in parts])
    stabs = np.concatenate([r[3] for r in parts])
    rnds  = np.concatenate([r[4] for r in parts])

    np.savez_compressed(cache_path, Ein=Ein, Edin=Edin,
                        labels=lbls, stabs=stabs, rnds=rnds)
    print(f"  {len(lbls):,} blocks in {time.time()-t0:.1f}s")
    return Ein, Edin, lbls, stabs, rnds


def balance_classes(Ein, Edin, labels, rnds=None, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    i0  = np.where(labels == 0)[0]
    i1  = np.where(labels == 1)[0]
    n   = min(len(i0), len(i1))
    idx = np.concatenate([rng.choice(i0, n, replace=False),
                           rng.choice(i1, n, replace=False)])
    rng.shuffle(idx)
    return Ein[idx], Edin[idx], labels[idx], (rnds[idx] if rnds is not None else None)


def normalize_inputs(Ein, Edin):
    def nm(X):
        m = X.mean(axis=0, keepdims=True)
        s = X.std(axis=0,  keepdims=True) + 1e-8
        return (X - m) / s, m, s
    En,  m1, s1 = nm(Ein)
    Edn, m2, s2 = nm(Edin)
    return En, Edn, (m1, s1, m2, s2)


def apply_norm(Ein, Edin, stats):
    m1, s1, m2, s2 = stats
    return (Ein - m1) / s1, (Edin - m2) / s2


print("Dataset utilities defined.")


Dataset utilities defined.


In [ ]:

#  MC-CNN ARCHITECTURE 


class JPEGDataset(Dataset):
    def __init__(self, Ein, Edin, labels):
        self.E  = torch.from_numpy(Ein.transpose(0, 3, 1, 2))   # (N,3,8,8)
        self.Ed = torch.from_numpy(Edin.transpose(0, 3, 1, 2))
        self.y  = torch.from_numpy(labels).float()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.E[i], self.Ed[i], self.y[i]


class DenseBlock(nn.Module):
    def __init__(self, in_ch, out_ch=12):
        super().__init__()
        self.c1 = nn.Sequential(
            nn.Conv2d(in_ch,          
            out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), 
            nn.LeakyReLU(0.2, True))
        self.c2 = nn.Sequential(
            nn.Conv2d(in_ch + out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), 
            nn.LeakyReLU(0.2, True))
        

    def forward(self, x):
        o1 = self.c1(x)
        o2 = self.c2(torch.cat([x, o1], 1))
        return torch.cat([x, o1, o2], 1)          


class Branch(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = DenseBlock(3, 12)
        self.c1x1  = nn.Sequential(nn.Conv2d(27, 12, 1), 
        nn.BatchNorm2d(12), nn.LeakyReLU(0.2, True))
        self.pool  = nn.AvgPool2d(2)              
        self.fc    = nn.Sequential(
            nn.Linear(12*4*4, 32),
            nn.BatchNorm1d(32), 
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.5))
    def forward(self, x):
        return self.fc(self.pool(self.c1x1(self.dense(x))).flatten(1))


class MCCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.bE   = Branch()
        self.bEd  = Branch()
        self.head = nn.Sequential(
            nn.Linear(64, 24), 
            nn.BatchNorm1d(24), 
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.5), 
            nn.Linear(24, 1), 
            nn.Sigmoid())
    def forward(self, E, Ed):
        return self.head(torch.cat([self.bE(E), self.bEd(Ed)], 1)).squeeze(1)


total_params = sum(p.numel() for p in MCCNN().parameters())
print(f"MC-CNN  —  total parameters: {total_params:,}")


MC-CNN  —  total parameters: 18,865


In [ ]:
#  TRAINING  &  EVALUATION  
def _loader(ds, shuffle=True):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0,
                      pin_memory=(DEVICE.type == 'cuda'))


def compute_metrics(labels, preds):
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn + 1e-10) * 100
    tnr = tn / (tn + fp + 1e-10) * 100
    return float(tpr), float(tnr), float((tpr + tnr) / 2)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, truths = [], []
    for E, Ed, y in loader:
        p = model(E.to(DEVICE), Ed.to(DEVICE)).cpu().numpy()
        preds.extend((p > 0.5).astype(int).tolist())
        truths.extend(y.numpy().astype(int).tolist())
    return compute_metrics(np.array(truths), np.array(preds))


def train_model(tr_ds, va_ds, ckpt_path=None, seed=0):    
    torch.manual_seed(seed)
    model = MCCNN().to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=LEARNING_RATE,
                       betas=(0.9, 0.999), weight_decay=1e-4)
    crit  = nn.BCELoss()
    start_epoch, best_bac, best_state = 1, -1.0, None

    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt'])
        start_epoch = ck['epoch'] + 1
        best_bac    = ck['best_bac']
        best_state  = ck['best_state']
        print(f"    Resumed ep {start_epoch-1}, best_bac={best_bac:.2f}")

    tr_ld = _loader(tr_ds, True)
    va_ld = _loader(va_ds, False)

    for ep in range(start_epoch, EPOCHS + 1):
        model.train()
        for E, Ed, y in tr_ld:
            E, Ed, y = E.to(DEVICE), Ed.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            crit(model(E, Ed), y).backward()
            opt.step()
        _, _, bac = evaluate(model, va_ld)
        if bac > best_bac:
            best_bac  = bac
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if ckpt_path and ep % 5 == 0:
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'epoch': ep, 'best_bac': best_bac,
                        'best_state': best_state}, ckpt_path)

    model.load_state_dict(best_state)
    return model, best_bac


#  best-model registry 
_REG_PATH = os.path.join(BEST_MODELS_DIR, "registry.json")

def _load_reg():
    return json.load(open(_REG_PATH)) if os.path.exists(_REG_PATH) else {}

def _save_reg(reg):
    json.dump(reg, open(_REG_PATH, 'w'), indent=2)

def save_best_model(model, bac, tag, q, norm_stats):
    reg = _load_reg()
    key = f"{tag}_q{q}"
    if bac > reg.get(key, {}).get('bac', -1):
        path = os.path.join(BEST_MODELS_DIR, f"{key}.pth")
        torch.save({
            'model_state': model.state_dict(),
            'bac': bac, 'tag': tag, 'quality_factor': q,
            'norm_m1': norm_stats[0].tolist(), 'norm_s1': norm_stats[1].tolist(),
            'norm_m2': norm_stats[2].tolist(), 'norm_s2': norm_stats[3].tolist(),
        }, path)
        reg[key] = {'bac': bac, 'path': path}
        _save_reg(reg)
        print(f"  ★ Best model saved → {path}  (BAC={bac:.2f})")


def load_best_model(tag, q):

    reg   = _load_reg()
    entry = reg.get(f"{tag}_q{q}")
    if not entry or not os.path.exists(entry['path']):
        return None, None
    ck = torch.load(entry['path'], map_location=DEVICE)
    m  = MCCNN().to(DEVICE)
    m.load_state_dict(ck['model_state'])
    m.eval()
    ns = tuple(np.array(ck[k]) for k in ('norm_m1','norm_s1','norm_m2','norm_s2'))
    return m, ns


def run_n_iterations(tr_Ein, tr_Edin, tr_labels,
                     te_Ein, te_Edin, te_labels,
                     ckpt_prefix, tag, q, norm_stats,
                     n_iter=NUM_ITERATIONS):
    
    res_path = ckpt_prefix + "_results.json"
    done = json.load(open(res_path)) if os.path.exists(res_path) else {}

    N, val_n = len(tr_labels), max(1, int(len(tr_labels) * 0.1))

    for it in range(n_iter):
        k = str(it)
        if k in done:
            print(f"    Iter {it+1}/{n_iter} [cached]  "
                  f"TPR={done[k][0]:.2f} TNR={done[k][1]:.2f} BAC={done[k][2]:.2f}")
            continue
        rng     = np.random.default_rng(it)
        idx     = rng.permutation(N)
        vi, ti  = idx[:val_n], idx[val_n:]
        tr_ds   = JPEGDataset(tr_Ein[ti],  tr_Edin[ti],  tr_labels[ti])
        va_ds   = JPEGDataset(tr_Ein[vi],  tr_Edin[vi],  tr_labels[vi])
        te_ds   = JPEGDataset(te_Ein,       te_Edin,       te_labels)
        ep_ckpt = ckpt_prefix + f"_it{it}.pth"
        model, val_bac = train_model(tr_ds, va_ds, ckpt_path=ep_ckpt, seed=it)
        tpr, tnr, bac  = evaluate(model, _loader(te_ds, False))
        save_best_model(model, bac, tag, q, norm_stats)
        done[k] = [tpr, tnr, bac]
        json.dump(done, open(res_path, 'w'))
        if os.path.exists(ep_ckpt):
            os.remove(ep_ckpt)
        print(f"    Iter {it+1}/{n_iter}  TPR={tpr:.2f} TNR={tnr:.2f} BAC={bac:.2f}")

    avg = lambda i: float(np.mean([done[str(k)][i] for k in range(n_iter)]))
    return avg(0), avg(1), avg(2)


print("Training utilities defined.")


Training utilities defined.


In [ ]:

#  EBSF (Yang et al.)


def ebsf_features(Ein, Edin):
    X = []
    for blk in [Ein[:, :, :, 0], Edin[:, :, :, 0]]:
        f = blk.reshape(len(blk), -1).astype(np.float32)
        X += [f.mean(1), f.var(1), np.abs(f).mean(1), (f**2).sum(1),
              f.max(1), f.min(1), np.median(f, 1),
              (f > 0).sum(1).astype(float),
              (f < 0).sum(1).astype(float),
              (f == 0).sum(1).astype(float)]
    return np.stack(X, axis=1)


def train_evaluate_ebsf(tr_Ein, tr_Edin, tr_labels,
                         te_Ein, te_Edin, te_labels):
    Xtr = ebsf_features(tr_Ein, tr_Edin)
    Xte = ebsf_features(te_Ein, te_Edin)
    sc  = StandardScaler().fit(Xtr)
    clf = LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_SEED)
    clf.fit(sc.transform(Xtr), tr_labels)
    return compute_metrics(te_labels, clf.predict(sc.transform(Xte)))


print("EBSF baseline defined.")


EBSF baseline defined.


In [ ]:

# 6. PLOTTING 


def _savefig(fig, name):
    p = os.path.join(PLOTS_DIR, name)
    fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved {p}")


def plot_figure2(test_paths):
    """Fig 2 — rounding / truncation block counts across quality factors."""
    out = os.path.join(PLOTS_DIR, "figure2.png")
    if os.path.exists(out):
        print(f"  [cached] {out}"); return

    counts = {q: {'sr': 0, 'st': 0, 'dr': 0, 'dt': 0} for q in QUALITY_FACTORS_DETAIL}
    for q in QUALITY_FACTORS_DETAIL:
        _, _, test_labels, stabs, rnds = build_dataset_cached(test_paths, q, "test")
        mask = stabs >= 1
        for lab, rnd in zip(test_labels[mask], rnds[mask]):
            key_prefix = 's' if lab == 0 else 'd'
            key_suffix = 'r' if rnd else 't'
            counts[q][key_prefix + key_suffix] += 1

    QFs = QUALITY_FACTORS_DETAIL
    x   = np.arange(len(QFs));  w = 0.2
    fig, ax = plt.subplots(figsize=(9, 5))
    for k, label, color, off in [
            ('sr', 'Single Rounding',    '#1f77b4', -1.5),
            ('st', 'Single Truncation',  '#ff7f0e', -0.5),
            ('dr', 'Double Rounding',    '#2ca02c',  0.5),
            ('dt', 'Double Truncation',  '#d62728',  1.5)]:
        ax.bar(x + off*w, [counts[q][k]/1e4 for q in QFs], w, label=label, color=color)
    ax.set_xlabel("Quality Factor"); ax.set_ylabel("Blocks (×10⁴)")
    ax.set_title("Fig 2. Rounding & Truncation Error Blocks (s ≥ 1)")
    ax.set_xticks(x); ax.set_xticklabels([f"q={q}" for q in QFs])
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    _savefig(fig, "figure2.png")


def plot_comparison(results, title, filename):
    QFs = sorted(results.keys())
    x   = np.arange(len(QFs))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric, mi in zip(axes, ("TPR", "TNR", "BAC"), range(3)):
        ax.bar(x-0.2, [results[q]['proposed'][mi] for q in QFs], 0.35,
               label='Proposed MC-CNN', color='steelblue')
        ax.bar(x+0.2, [results[q]['ebsf'][mi]     for q in QFs], 0.35,
               label='EBSF [7]', color='coral')
        ax.set_title(metric); ax.set_xticks(x); ax.set_xticklabels([str(q) for q in QFs])
        ax.set_xlabel("Quality Factor"); ax.set_ylabel(f"{metric} (%)")
        ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 100)
    fig.suptitle(title, fontweight='bold'); plt.tight_layout()
    _savefig(fig, filename)


def plot_table1_bars(results_t1):
    QFs = sorted(results_t1.keys())
    x   = np.arange(len(QFs))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric, mi in zip(axes, ("TPR", "TNR", "BAC"), range(3)):
        ax.bar(x-0.2, [results_t1[q]['rounding'][mi]   for q in QFs], 0.35,
               label='Rounding',   color='steelblue')
        ax.bar(x+0.2, [results_t1[q]['truncation'][mi] for q in QFs], 0.35,
               label='Truncation', color='darkorange')
        ax.set_title(metric); ax.set_xticks(x); ax.set_xticklabels([str(q) for q in QFs])
        ax.set_xlabel("Quality Factor"); ax.set_ylabel(f"{metric} (%)")
        ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 100)
    fig.suptitle("Table 1: Rounding vs Truncation Blocks (s ≥ 1)", fontweight='bold')
    plt.tight_layout(); _savefig(fig, "table1_rounding_truncation.png")


def plot_table4_heatmap(mat, QFs):
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(mat, cmap='RdYlGn', vmin=55, vmax=85)
    ax.set_xticks(range(len(QFs))); ax.set_xticklabels(QFs)
    ax.set_yticks(range(len(QFs))); ax.set_yticklabels(QFs)
    ax.set_xlabel("Test QF"); ax.set_ylabel("Train QF")
    ax.set_title("Table 4: Cross-QF Balanced Accuracy (%)")
    for i in range(len(QFs)):
        for j in range(len(QFs)):
            ax.text(j, i, f"{mat[i,j]:.1f}", ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=ax, label='BAC (%)')
    plt.tight_layout(); _savefig(fig, "table4_cross_qf_heatmap.png")


def print_table(title, results):
    print(f"\n{'='*74}\n{title}\n{'='*74}")
    ks = sorted(results.keys())
    if 'proposed' in results[ks[0]]:
        print(f"{'QF':>4} | {'── Proposed MC-CNN ──':^29} | {'── EBSF [7] ──':^29}")
        print(f"{'':>4} | {'TPR':>9} {'TNR':>9} {'BAC':>9} | {'TPR':>9} {'TNR':>9} {'BAC':>9}")
        for q in ks:
            a,b,c = results[q]['proposed']; d,e,f = results[q]['ebsf']
            print(f"{q:>4} | {a:>9.2f} {b:>9.2f} {c:>9.2f} | {d:>9.2f} {e:>9.2f} {f:>9.2f}")
    else:
        print(f"{'QF':>4} | {'── Rounding ──':^29} | {'── Truncation ──':^29}")
        print(f"{'':>4} | {'TPR':>9} {'TNR':>9} {'BAC':>9} | {'TPR':>9} {'TNR':>9} {'BAC':>9}")
        for q in ks:
            a,b,c = results[q]['rounding']; d,e,f = results[q]['truncation']
            print(f"{q:>4} | {a:>9.2f} {b:>9.2f} {c:>9.2f} | {d:>9.2f} {e:>9.2f} {f:>9.2f}")


print("Plotting helpers defined.")


Plotting helpers defined.


In [ ]:

# LOAD IMAGE PATHS  (UCID — 1338 .tif images)

def get_image_paths():
    files = sorted(f for f in os.listdir(DATASET_PATH)
                   if f.lower().endswith(('.tif', '.tiff')))
    paths = [os.path.join(DATASET_PATH, f) for f in files]
    if len(paths) < 1338:
        raise ValueError(f"Need ≥1338 .tif images, found {len(paths)} in '{DATASET_PATH}'")
    print(f"Found {len(paths)} images.  Train=669  Test=669")
    return paths[:669], paths[669:1338]

train_paths, test_paths = get_image_paths()


Found 1338 images.  Train=669  Test=669


In [ ]:
#  FIGURE 2  — Rounding & Truncation Block Distribution
print(">>> Figure 2 ...")
plot_figure2(test_paths)
print("Done.")


>>> Figure 2 ...
  [cached] purvaji\plots_fast_exact_new\figure2.png
Done.


In [ ]:

# 9. TABLE 3  — All Distinguishable Blocks  (s ≥ 1)   


def run_table3():
    res_path = os.path.join(CHECKPOINT_DIR, "results_table3.json")
    results  = json.load(open(res_path)) if os.path.exists(res_path) else {}
    print("\n" + "="*74 + "\nTABLE 3: All distinguishable blocks (s ≥ 1)\n" + "="*74)

    for q in QUALITY_FACTORS_MAIN:
        sq = str(q)
        if sq in results:
            r = results[sq]
            print(f"  q={q:2d} [cached] Prop BAC={r['proposed'][2]:.2f}  EBSF BAC={r['ebsf'][2]:.2f}")
            continue
        print(f"\n  q={q}")
        tr_Ein,tr_Edin,tr_l,_,_ = build_dataset_cached(train_paths, q, "train")
        te_Ein,te_Edin,te_l,_,_ = build_dataset_cached(test_paths,  q, "test")
        tr_E,tr_Ed,tr_lb,_ = balance_classes(tr_Ein, tr_Edin, tr_l)
        tr_En,tr_Edn,stats = normalize_inputs(tr_E, tr_Ed)
        te_En,te_Edn       = apply_norm(te_Ein, te_Edin, stats)
        pfx = os.path.join(CHECKPOINT_DIR, f"t3_q{q}")
        tpr_p,tnr_p,bac_p = run_n_iterations(tr_En,tr_Edn,tr_lb,
                                               te_En,te_Edn,te_l,
                                               pfx,"table3",q,stats)
        tpr_e,tnr_e,bac_e = train_evaluate_ebsf(tr_E,tr_Ed,tr_lb,
                                                  te_Ein,te_Edin,te_l)
        results[sq] = {'proposed':[tpr_p,tnr_p,bac_p], 'ebsf':[tpr_e,tnr_e,bac_e]}
        json.dump(results, open(res_path,'w'), indent=2)

    disp = {int(k): v for k,v in results.items()}
    print_table("TABLE 3 Final Results", disp)
    plot_comparison(disp, "Table 3: All Distinguishable Blocks (s ≥ 1)", "table3_all_blocks.png")
    return disp

results_t3 = run_table3()



TABLE 3: All distinguishable blocks (s ≥ 1)
  q=20 [cached] Prop BAC=75.89  EBSF BAC=61.80
  q=40 [cached] Prop BAC=76.63  EBSF BAC=62.29
  q=60 [cached] Prop BAC=76.07  EBSF BAC=62.11
  q=70 [cached] Prop BAC=74.92  EBSF BAC=62.72
  q=75 [cached] Prop BAC=74.54  EBSF BAC=62.48
  q=80 [cached] Prop BAC=80.71  EBSF BAC=73.12
  q=90 [cached] Prop BAC=82.87  EBSF BAC=78.18

TABLE 3 Final Results
  QF |     ── Proposed MC-CNN ──     |        ── EBSF [7] ──        
     |       TPR       TNR       BAC |       TPR       TNR       BAC
  20 |     77.54     74.24     75.89 |     52.19     71.41     61.80
  40 |     82.74     70.52     76.63 |     66.50     58.07     62.29
  60 |     84.77     67.37     76.07 |     71.94     52.28     62.11
  70 |     86.10     63.74     74.92 |     75.08     50.36     62.72
  75 |     85.03     64.04     74.54 |     76.54     48.43     62.48
  80 |     90.32     71.09     80.71 |     92.13     54.12     73.12
  90 |     93.97     71.78     82.87 |     90.94   

In [ ]:

# 10. TABLE 2  — Blocks with Stability Index s = 1 

def run_table2():
    res_path = os.path.join(CHECKPOINT_DIR, "results_table2.json")
    results  = json.load(open(res_path)) if os.path.exists(res_path) else {}
    print("\n" + "="*74 + "\nTABLE 2: Blocks with stability index s = 1\n" + "="*74)

    for q in QUALITY_FACTORS_DETAIL:
        sq = str(q)
        if sq in results:
            r = results[sq]
            print(f"  q={q:2d} [cached] Prop BAC={r['proposed'][2]:.2f}  EBSF BAC={r['ebsf'][2]:.2f}")
            continue
        print(f"\n  q={q}")
        tr_Ein,tr_Edin,tr_l,tr_s,_ = build_dataset_cached(train_paths, q, "train")
        te_Ein,te_Edin,te_l,te_s,_ = build_dataset_cached(test_paths,  q, "test")

        # Filter s=1 only
        tr_Ein,tr_Edin,tr_l = tr_Ein[tr_s==1], tr_Edin[tr_s==1], tr_l[tr_s==1]
        te_Ein,te_Edin,te_l = te_Ein[te_s==1], te_Edin[te_s==1], te_l[te_s==1]

        # s=1: E_{n+2} == E_{n+1}, duplicate channel 1 into channel 2
        for arr in (tr_Ein, tr_Edin, te_Ein, te_Edin):
            arr[:,:,:,2] = arr[:,:,:,1]

        tr_E,tr_Ed,tr_lb,_ = balance_classes(tr_Ein, tr_Edin, tr_l)
        tr_En,tr_Edn,stats = normalize_inputs(tr_E, tr_Ed)
        te_En,te_Edn       = apply_norm(te_Ein, te_Edin, stats)
        pfx = os.path.join(CHECKPOINT_DIR, f"t2_q{q}")
        tpr_p,tnr_p,bac_p = run_n_iterations(tr_En,tr_Edn,tr_lb,
                                               te_En,te_Edn,te_l,
                                               pfx,"table2",q,stats)
        tpr_e,tnr_e,bac_e = train_evaluate_ebsf(tr_E,tr_Ed,tr_lb,
                                                  te_Ein,te_Edin,te_l)
        results[sq] = {'proposed':[tpr_p,tnr_p,bac_p], 'ebsf':[tpr_e,tnr_e,bac_e]}
        json.dump(results, open(res_path,'w'), indent=2)

    disp = {int(k): v for k,v in results.items()}
    print_table("TABLE 2 Final Results", disp)
    plot_comparison(disp, "Table 2: Blocks with s = 1", "table2_s1_blocks.png")
    return disp

results_t2 = run_table2()



TABLE 2: Blocks with stability index s = 1
  q=60 [cached] Prop BAC=73.45  EBSF BAC=61.89
  q=70 [cached] Prop BAC=71.89  EBSF BAC=62.02
  q=75 [cached] Prop BAC=70.53  EBSF BAC=61.78
  q=80 [cached] Prop BAC=83.99  EBSF BAC=81.34
  q=85 [cached] Prop BAC=85.49  EBSF BAC=83.51
  q=90 [cached] Prop BAC=88.64  EBSF BAC=86.23

TABLE 2 Final Results
  QF |     ── Proposed MC-CNN ──     |        ── EBSF [7] ──        
     |       TPR       TNR       BAC |       TPR       TNR       BAC
  60 |     81.46     65.43     73.45 |     69.02     54.76     61.89
  70 |     80.80     62.99     71.89 |     69.65     54.38     62.02
  75 |     80.50     60.56     70.53 |     71.00     52.56     61.78
  80 |     93.86     74.12     83.99 |     97.41     65.27     81.34
  85 |     95.90     75.08     85.49 |     98.68     68.34     83.51
  90 |     96.77     80.51     88.64 |     90.67     81.78     86.23
  Saved purvaji\plots_fast_exact_new\table2_s1_blocks.png


In [ ]:
# 11. TABLE 1  — Rounding vs Truncation Error Block Analysis

def run_table1():
    res_path = os.path.join(CHECKPOINT_DIR, "results_table1.json")
    results  = json.load(open(res_path)) if os.path.exists(res_path) else {}
    print("\n" + "="*74 + "\nTABLE 1: Rounding vs Truncation (s ≥ 1)\n" + "="*74)

    for q in QUALITY_FACTORS_DETAIL:
        sq = str(q)
        if sq in results:
            r = results[sq]
            print(f"  q={q:2d} [cached] Rnd BAC={r['rounding'][2]:.2f}  Trn BAC={r['truncation'][2]:.2f}")
            continue
        print(f"\n  q={q}")
        tr_Ein,tr_Edin,tr_l,_,tr_rnd = build_dataset_cached(train_paths, q, "train")
        te_Ein,te_Edin,te_l,_,te_rnd = build_dataset_cached(test_paths,  q, "test")

        tr_E,tr_Ed,tr_lb,_ = balance_classes(tr_Ein, tr_Edin, tr_l)
        tr_En,tr_Edn,stats = normalize_inputs(tr_E, tr_Ed)
        te_En,te_Edn       = apply_norm(te_Ein, te_Edin, stats)

       
        best_bac, best_model = -1.0, None
        N, val_n = len(tr_lb), max(1, int(len(tr_lb)*0.1))
        for it in range(NUM_ITERATIONS):
            rng    = np.random.default_rng(it)
            idx    = rng.permutation(N)
            vi, ti = idx[:val_n], idx[val_n:]
            tr_ds  = JPEGDataset(tr_En[ti], tr_Edn[ti], tr_lb[ti])
            va_ds  = JPEGDataset(tr_En[vi], tr_Edn[vi], tr_lb[vi])
            ck_p   = os.path.join(CHECKPOINT_DIR, f"t1_q{q}_it{it}.pth")
            m, _   = train_model(tr_ds, va_ds, ckpt_path=ck_p, seed=it)
            _, _, vbac = evaluate(m, _loader(va_ds, False))
            print(f"    Iter {it+1}/{NUM_ITERATIONS}  val_BAC={vbac:.2f}")
            if vbac > best_bac:
                best_bac, best_model = vbac, m
            if os.path.exists(ck_p): os.remove(ck_p)

        save_best_model(best_model, best_bac, "table1", q, stats)

        def eval_sub(mask):
            if not mask.any(): return 0., 0., 0.
            ds = JPEGDataset(te_En[mask], te_Edn[mask], te_l[mask])
            return evaluate(best_model, _loader(ds, False))

        tpr_r,tnr_r,bac_r = eval_sub(te_rnd)
        tpr_t,tnr_t,bac_t = eval_sub(~te_rnd)
        results[sq] = {'rounding':   [tpr_r,tnr_r,bac_r],
                       'truncation': [tpr_t,tnr_t,bac_t]}
        json.dump(results, open(res_path,'w'), indent=2)

    disp = {int(k): v for k,v in results.items()}
    print_table("TABLE 1 Final Results", disp)
    plot_table1_bars(disp)
    return disp

results_t1 = run_table1()



TABLE 1: Rounding vs Truncation (s ≥ 1)
  q=60 [cached] Rnd BAC=0.00  Trn BAC=76.45
  q=70 [cached] Rnd BAC=50.00  Trn BAC=75.04
  q=75 [cached] Rnd BAC=68.53  Trn BAC=74.58
  q=80 [cached] Rnd BAC=50.00  Trn BAC=72.25
  q=85 [cached] Rnd BAC=50.00  Trn BAC=70.86
  q=90 [cached] Rnd BAC=82.78  Trn BAC=66.04

TABLE 1 Final Results
  QF |        ── Rounding ──         |       ── Truncation ──       
     |       TPR       TNR       BAC |       TPR       TNR       BAC
  60 |      0.00      0.00      0.00 |     85.71     67.20     76.45
  70 |      0.00    100.00     50.00 |     85.99     64.09     75.04
  75 |     40.00     97.06     68.53 |     83.73     65.43     74.58
  80 |      0.00    100.00     50.00 |     89.65     54.85     72.25
  85 |      0.00    100.00     50.00 |     91.40     50.33     70.86
  90 |     66.14     99.43     82.78 |     94.29     37.78     66.04
  Saved purvaji\plots_fast_exact_new\table1_rounding_truncation.png


In [ ]:
# 12. TABLE 4  — Cross Quality-Factor Detection (Universal Detector)

def run_table4():
    res_path = os.path.join(CHECKPOINT_DIR, "results_table4.json")
    results  = json.load(open(res_path)) if os.path.exists(res_path) else {}
    QFs      = QUALITY_FACTORS_DETAIL
    print("\n" + "="*74 + "\nTABLE 4: Cross-QF detection\n" + "="*74)

    # Pre-load all test caches
    print("  Pre-loading test caches ...")
    test_data = {}
    for q in QFs:
        te_Ein,te_Edin,te_l,_,_ = build_dataset_cached(test_paths, q, "test")
        test_data[q] = (te_Ein, te_Edin, te_l)

    for q_train in QFs:
        tr_Ein,tr_Edin,tr_l,_,_ = build_dataset_cached(train_paths, q_train, "train")
        tr_E,tr_Ed,tr_lb,_ = balance_classes(tr_Ein, tr_Edin, tr_l)
        tr_En,tr_Edn,stats = normalize_inputs(tr_E, tr_Ed)

        for q_test in QFs:
            key = f"{q_train}_{q_test}"
            if key in results:
                print(f"  train={q_train} test={q_test} [cached] BAC={results[key]:.2f}")
                continue
            te_Ein,te_Edin,te_l = test_data[q_test]
            te_En,te_Edn = apply_norm(te_Ein, te_Edin, stats)
            pfx  = os.path.join(CHECKPOINT_DIR, f"t4_{q_train}_{q_test}")
            _,_,bac = run_n_iterations(tr_En,tr_Edn,tr_lb,
                                        te_En,te_Edn,te_l,
                                        pfx, f"table4_{q_train}", q_test, stats)
            results[key] = bac
            json.dump(results, open(res_path,'w'), indent=2)
            print(f"  train={q_train} test={q_test}  BAC={bac:.2f}")

    bac_mat = np.array([[results.get(f"{qt}_{qe}", 0.) for qe in QFs] for qt in QFs])
    print("\nTable 4 (BAC %):")
    print(f"{'↓train/test→':>14}" + "".join(f"{q:>8}" for q in QFs))
    for i,qt in enumerate(QFs):
        print(f"{qt:>14}" + "".join(f"{bac_mat[i,j]:>8.2f}" for j in range(len(QFs))))
    plot_table4_heatmap(bac_mat, QFs)
    return bac_mat, QFs

bac_mat, QFs_detail = run_table4()



TABLE 4: Cross-QF detection
  Pre-loading test caches ...
  [cache] purvaji\checkpoints_fast_exact_new\cache_test_q60.npz
  [cache] purvaji\checkpoints_fast_exact_new\cache_test_q70.npz
  [cache] purvaji\checkpoints_fast_exact_new\cache_test_q75.npz
  [cache] purvaji\checkpoints_fast_exact_new\cache_test_q80.npz
  [cache] purvaji\checkpoints_fast_exact_new\cache_test_q85.npz
  [cache] purvaji\checkpoints_fast_exact_new\cache_test_q90.npz
  [cache] purvaji\checkpoints_fast_exact_new\cache_train_q60.npz
  train=60 test=60 [cached] BAC=76.07
  train=60 test=70 [cached] BAC=69.32
  train=60 test=75 [cached] BAC=63.52
  train=60 test=80 [cached] BAC=72.03
  train=60 test=85 [cached] BAC=71.22
  train=60 test=90 [cached] BAC=69.03
  [cache] purvaji\checkpoints_fast_exact_new\cache_train_q70.npz
  train=70 test=60 [cached] BAC=67.77
  train=70 test=70 [cached] BAC=74.92
  train=70 test=75 [cached] BAC=71.81
  train=70 test=80 [cached] BAC=74.27
  train=70 test=85 [cached] BAC=70.77
  train=7

In [ ]:
#  SUMMARY

reg = _load_reg()
print("\n" + "="*74)
print("ALL EXPERIMENTS COMPLETE")
print(f"  Plots      → {PLOTS_DIR}/")
print(f"  Caches     → {CHECKPOINT_DIR}/")
print(f"  Best models→ {BEST_MODELS_DIR}/")
print("="*74)
if reg:
    print("\nBest model registry:")
    for key, entry in sorted(reg.items()):
        print(f"  {key:<32}  BAC={entry['bac']:.2f}  →  {entry['path']}")
else:
    print("(No experiments completed yet)")



ALL EXPERIMENTS COMPLETE
  Plots      → purvaji\plots_fast_exact_new/
  Caches     → purvaji\checkpoints_fast_exact_new/
  Best models→ purvaji\best_models_fast_exact_new/

Best model registry:
  table1_q60                        BAC=75.67  →  purvaji\best_models_fast_exact_new\table1_q60.pth
  table1_q70                        BAC=74.61  →  purvaji\best_models_fast_exact_new\table1_q70.pth
  table1_q75                        BAC=74.41  →  purvaji\best_models_fast_exact_new\table1_q75.pth
  table1_q80                        BAC=79.58  →  purvaji\best_models_fast_exact_new\table1_q80.pth
  table1_q85                        BAC=81.58  →  purvaji\best_models_fast_exact_new\table1_q85.pth
  table1_q90                        BAC=82.21  →  purvaji\best_models_fast_exact_new\table1_q90.pth
  table2_q60                        BAC=73.82  →  purvaji\best_models_fast_exact_new\table2_q60.pth
  table2_q70                        BAC=72.27  →  purvaji\best_models_fast_exact_new\table2_q70.pth
  tab